# CU 신상품 키워드 추출 결과 비교 분석 (필터링: 누락/속성 부족)

**분석 목표:** `CU_single_products_new.xlsx` 파일에서 **'비고'** 컬럼이 **'누락'** 또는 **'속성 부족'**인 데이터만 추출하여, 기존 결과 대비 신규(V2) 추출 결과가 얼마나 개선되었는지 검토합니다.
- **대상:** 비고 == '누락' 또는 '속성 부족'
- **기존 데이터:** `keywords_json` 열
- **신규 데이터:** 맨 우측 열 (V2 방식 - 상품별 구조화)

In [28]:
import pandas as pd
import json
import re
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from IPython.display import display

# 한글 폰트 설정
try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
except:
    pass
plt.rcParams['axes.unicode_minus'] = False

print("분석 환경 준비 완료")

분석 환경 준비 완료


### 1. 데이터 로드 및 필터링

In [29]:
file_path = '../data/processed/CU_single_products_new.xlsx'
df_raw = pd.read_excel(file_path)

target_remarks = ['누락', '속성 부족']
df = df_raw[df_raw['비고'].isin(target_remarks)].copy()

old_col = 'keywords_json'
new_col = df.columns[-1]

print(f"필터링 후 분석 대상 행 수: {len(df)}")

필터링 후 분석 대상 행 수: 29


### 2. 상품별 구조화 파싱 함수

In [30]:
def parse_structured_keywords(json_input):
    """JSON에서 상품별로 구분된 키워드 문자열을 반환합니다."""
    if pd.isna(json_input) or str(json_input).strip() == '':
        return "-"
    
    try:
        data = json_input
        if isinstance(json_input, str):
            json_str = json_input.strip()
            start = json_str.find('{')
            end = json_str.rfind('}') + 1
            if start != -1 and end != 0:
                data = json.loads(json_str[start:end])
            else:
                return "-"

        # V2 구조: metadata 리스트 순회
        if 'metadata' in data and isinstance(data['metadata'], list):
            product_outputs = []
            for item in data['metadata']:
                p_name = item.get('name', '알 수 없는 상품')
                p_kws = []
                for key in ['flavor_and_category', 'collab_and_brand', 'promotion_type', 'tpo_context']:
                    if key in item and isinstance(item[key], list):
                        p_kws.extend(item[key])
                
                if p_kws:
                    product_outputs.append(f"[{p_name}] {', '.join(p_kws)}")
                else:
                    product_outputs.append(f"[{p_name}] 키워드 없음")
            
            return "\n".join(product_outputs)
        
        # V1 구조: 루트 레벨 키워드
        all_kws = []
        for key in ['flavor_and_category', 'collab_and_brand', 'promotion_type', 'tpo_context']:
            if key in data and isinstance(data[key], list):
                all_kws.extend(data[key])
        return ", ".join(all_kws) if all_kws else "-"
        
    except:
        return "파싱 오류"

def get_total_count(json_input):
    """통계용 총 키워드 개수 계산"""
    try:
        res = parse_structured_keywords(json_input)
        if res == "-" or res == "파싱 오류": return 0
        # 대괄호와 콤마를 제외한 순수 단어 개수 대략 추정 (또는 리스트 합산 로직 재사용)
        # 정확도를 위해 이전 로직의 리스트 합산을 내부적으로 수행
        data = json.loads(json_input) if isinstance(json_input, str) else json_input
        count = 0
        if 'metadata' in data and isinstance(data['metadata'], list):
            for item in data['metadata']:
                for k in ['flavor_and_category', 'collab_and_brand', 'promotion_type', 'tpo_context']:
                    if k in item and isinstance(item[k], list): count += len(item[k])
        for k in ['flavor_and_category', 'collab_and_brand', 'promotion_type', 'tpo_context']:
            if k in data and isinstance(data[k], list): count += len(data[k])
        return count
    except: return 0

df['old_kw_display'] = df[old_col].apply(parse_structured_keywords)
df['new_kw_display'] = df[new_col].apply(parse_structured_keywords)
df['old_count'] = df[old_col].apply(get_total_count)
df['new_count'] = df[new_col].apply(get_total_count)
df['increase'] = df['new_count'] - df['old_count']

### 3. 필터링 대상 상세 비교 테이블 (상품별 구조 확인)

개선 후 키워드(`new_kw_display`)가 **[상품명] 키워드...** 형태로 표시되어 여러 상품이 포함된 게시글을 명확히 구분합니다.

In [31]:
summary_df = pd.DataFrame({
    '게시글 제목(Title)': df['title'],
    '원본 본문(Body)': df['body'],
    '개선 전 키워드': df['old_kw_display'],
    '개선 후 키워드(상품별)': df['new_kw_display'],
    '증가량': df['increase']
})

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 300)

print(f"'누락/속성 부족' 샘플 상세 비교 결과 (총 {len(summary_df)}건)")
display(summary_df)

'누락/속성 부족' 샘플 상세 비교 결과 (총 29건)


,게시글 제목(Title),원본 본문(Body),개선 전 키워드,개선 후 키워드(상품별),증가량
48,NaN,딩동~♬\nCU덕후가 신상 리뷰를 공유합니다👀\n✨찐 CU덕후가 공유한 생생한 리뷰 확인해볼까요✨\n\n맛폴리XCU의 콜라보 파스타\n맛폴리쉬림프알리오올리오🍝\n\n전국팔도의 맛을 집에서 간편하게!\n팔도한끼여수식매콤낙지비빔밥🐙\n\n꾸덕하고 진한 소스가 듬뿍~\n신라면 툼바🍝\n\n왕 크니까 왕 맛있는 가성비 버거\n자이언트빅클래식버거🍔\n\n당 충전으로 최고의 디저트지!\n허쉬오리지널롤케잌🍫🍰\n\n짭짤고소한 팝콘 맛에 콰삭콰삭 바삭한 식감을 더한\n꼬북칩카라멜팝콘맛🥠\n\n🎁 #EVENT 🎁\n\n✨가장 취향 저격인 신...,"[맛폴리쉬림프알리오올리오] 파스타, 맛폴리, 간편식\n[팔도한끼여수식매콤낙지비빔밥] 비빔밥, 팔도\n[신라면 툼바] 라면, 신라면\n[자이언트빅클래식버거] 버거\n[허쉬오리지널롤케잌] 케이크, 허쉬, 디저트\n[꼬북칩카라멜팝콘맛] 과자, 꼬북칩","[맛폴리쉬림프알리오올리오] 쉬림프, 새우, 알리오올리오, 파스타, 오일파스타, 콜라보파스타, 맛폴리, 콜라보, 신상, 댓글이벤트, 간편식, 한끼\n[팔도한끼여수식매콤낙지비빔밥] 매콤, 낙지, 비빔밥, 지역식, 여수식, 간편식, 팔도, 팔도한끼, 콜라보, 신상, 댓글이벤트, 한끼, 간편식, 집밥\n[신라면툼바] 신라면, 툼바, 크림, 꾸덕, 진한소스, 라면, 농심, 신라면, 콜라보, 신상, 댓글이벤트, 야식, 한끼\n[자이언트빅클래식버거] 빅사이즈, 클래식, 버거, 햄버거, 신상, 가성비, 댓글이벤트, 한끼, 간편식\n[허쉬오...",60
63,ㅤ,"。.。:+* ゜ ゜゜ *+:。.。:+* ゜ ゜゜ *+:。.。.。:+* ゜ ゜゜ \n🍰 SWEET & PETIT DESSERT HOUSE 당과점 🍰\n\n낰낰! 소문 듣고 왔어요\n여기가 유우-명한 디저트 베이커리인가요?\nㄴ 네! ✺◟(∗❛ัᴗ❛ั∗)◞✺\n\n보기만 해도 달콤해지는 초호화 라인업 ⸝⁺⊹\n\n🍪 촉촉한 초코와 꾸덕한 브라우니의 만남 #쫀덕케\n🧁 입안 가득 차는 뚠뚠한 행복 #뚱카롱\n🧁 폭신한 다쿠아즈, 아낌없는 크림 #뚱쿠아즈\n🍰 초콜릿 코팅 속 부드러운 케익이 쏙! #볼케익\n🧈 꾸덕 쫀득의 정석!...","[쫀덕케] 초코, 말차, 브라우니, 쫀득, 케이크, 당충전\n[뚱카롱] 베리, 커피, 크림, 디저트, 당충전\n[뚱쿠아즈] 다쿠아즈, 크림, 당충전","[당과점)초코무스쫀덕케] 초코, 무스, 브라우니, 쫀득, 꾸덕, 촉촉, 케이크, 당과점, 댓글이벤트, 단독, 당충전, 디저트\n[당과점)말차무스쫀덕케] 말차, 무스, 브라우니, 쫀득, 케이크, 당과점, 댓글이벤트, 단독, 당충전, 디저트\n[당과점)베리뚱카롱] 베리, 크림, 마카롱, 당과점, 댓글이벤트, 단독, 당충전, 디저트\n[당과점)커피뚱카롱] 커피, 크림, 마카롱, 당과점, 댓글이벤트, 단독, 당충전, 디저트\n[당과점)더블뚱쿠아즈] 다쿠아즈, 크림, 폭신, 당과점, 댓글이벤트, 단독, 당충전, 디저트\n[당과점)밀크...",66
64,ㅤ,"。.。:+* ゜ ゜゜ *+:。.。:+* ゜ ゜゜ *+:。.。.。:+* ゜ ゜゜ \n🍰 SWEET & PETIT DESSERT HOUSE 당과점 🍰\n\n낰낰! 소문 듣고 왔어요\n여기가 유우-명한 디저트 베이커리인가요?\nㄴ 네! ✺◟(∗❛ัᴗ❛ั∗)◞✺\n\n보기만 해도 달콤해지는 초호화 라인업 ⸝⁺⊹\n\n🍪 촉촉한 초코와 꾸덕한 브라우니의 만남 #쫀덕케\n🧁 입안 가득 차는 뚠뚠한 행복 #뚱카롱\n🧁 폭신한 다쿠아즈, 아낌없는 크림 #뚱쿠아즈\n🍰 초콜릿 코팅 속 부드러운 케익이 쏙! #볼케익\n🧈 꾸덕 쫀득의 정석!...","[쫀덕케] 초코, 말차, 브라우니, 쫀득, 케이크, 당충전\n[뚱카롱] 베리, 커피, 크림, 디저트, 당충전\n[뚱쿠아즈] 다쿠아즈, 크림, 당충전","[당과점)초코무스쫀덕케] 초코, 무스, 브라우니, 쫀득, 꾸덕, 촉촉, 케이크, 당과점, 댓글이벤트, 단독, 당충전, 디저트\n[당과점)말차무스쫀덕케] 말차, 무스, 브라우니, 쫀득, 케이크, 당과점, 댓글이벤트, 단독, 당충전, 디저트\n[당과점)베리뚱카롱] 베리, 크림, 마카롱, 당과점, 댓글이벤트, 단독, 당충전, 디저트\n[당과점)커피뚱카롱] 커피, 크림, 마카롱, 당과점, 댓글이벤트, 단독, 당충전, 디저트\n[당과점)더블뚱쿠아즈] 다쿠아즈, 크림, 폭신, 당과점, 댓글이벤트, 단독, 당충전, 디저트\n[당과점)밀크...",66
67,ㅤ,"🍽 𝙄‘𝙇𝙇 𝘽𝙀 𝘽𝘼𝘾𝙆 🍽\n\n10년간 뜨거운 사랑을 받았던 \n백종원 간편식 시리즈 10종이 출시되었습니다🔥\n\n푸짐하고 따끈한 그 맛!\n더 가성비 있게 즐기는 TIP! 💡\n \n⭐️10종 모두 40% 할인 ⭐️\n📌 할인 방법: 하나카드/Npay 결제 + QR코드 제시\n📌 할인 기간: 2/26(수) ~ 3/31(월)\n※ 자세한 내용은 포켓CU 이벤트 페이지를 참고해주세요!\n\n오직 CU에서 만나보세요💜\n도) 백종원스페셜한판 / 4,900원\n도) 백종원스페셜매콤불고기 / 4,900원\n주) 백종원스페셜우...","[백종원스페셜한판] 한식, 간편식, 백종원, 40% 할인, 집밥, 간편식\n[백종원스페셜매콤불고기] 매콤, 불고기, 백종원, 40% 할인, 집밥, 간편식","[도)백종원스페셜한판] 도시락, 한식, 백종원, 40% 할인, 콤보가/Npay결제, 댓글이벤트, 간편식, 따끈한식사\n[도)백종원스페셜매콤불고기] 도시락, 매콤, 불고기, 백종원, 40% 할인, 댓글이벤트, 간편식\n[주)백종원스페셜우삼격] 삼각김밥, 우삼격, 백종원, 40% 할인, 댓글이벤트, 간편식, 간식\n[주)백종원스페셜대파제육] 삼각김밥, 대파, 제육, 백종원, 40% 할인, 댓글이벤트, 간편식\n[김)백종원스페셜한줄김밥] 한줄김밥, 김밥, 백종원, 40% 할인, 댓글이벤트, 간편식\n[김)백종원스페셜매콤불고기] 김...",57
68,ㅤ,"🍽 𝙄‘𝙇𝙇 𝘽𝙀 𝘽𝘼𝘾𝙆 🍽\n\n10년간 뜨거운 사랑을 받았던 \n백종원 간편식 시리즈 10종이 출시되었습니다🔥\n\n푸짐하고 따끈한 그 맛!\n더 가성비 있게 즐기는 TIP! 💡\n \n⭐️10종 모두 40% 할인 ⭐️\n📌 할인 방법: 하나카드/Npay 결제 + QR코드 제시\n📌 할인 기간: 2/26(수) ~ 3/31(월)\n※ 자세한 내용은 포켓CU 이벤트 페이지를 참고해주세요!\n\n오직 CU에서 만나보세요💜\n도) 백종원스페셜한판 / 4,900원\n도) 백종원스페셜매콤불고기 / 4,900원\n주) 백종원스페셜우...","[백종원스페셜한판] 한식, 간편식, 백종원, 40% 할인, 집밥, 간편식\n[백종원스페셜매콤불고기] 매콤, 불고기, 백종원, 40% 할인, 집밥, 간편식","[도)백종원스페셜한판] 도시락, 한식, 백종원, 40% 할인, 콤보가/Npay결제, 댓글이벤트, 간편식, 따끈한식사\n[도)백종원스페셜매콤불고기] 도시락, 매콤, 불고기, 백종원, 40% 할인, 댓글이벤트, 간편식\n[주)백종원스페셜우삼격] 삼각김밥, 우삼격, 백종원, 40% 할인, 댓글이벤트, 간편식, 간식\n[주)백종원스페셜대파제육] 삼각김밥, 대파, 제육, 백종원, 40% 할인, 댓글이벤트, 간편식\n[김)백종원스페셜한줄김밥] 한줄김밥, 김밥, 백종원, 40% 할인, 댓글이벤트, 간편식\n[김)백종원스페셜매콤불고기] 김...",57
69,ㅤ,"🍽 𝙄‘𝙇𝙇 𝘽𝙀 𝘽𝘼𝘾𝙆 🍽\n\n10년간 뜨거운 사랑을 받았던 \n백종원 간편식 시리즈 10종이 출시되었습니다🔥\n\n푸짐하고 따끈한 그 맛!\n더 가성비 있게 즐기는 TIP! 💡\n \n⭐️10종 모두 40% 할인 ⭐️\n📌 할인 방법: 하나카드/Npay 결제 + QR코드 제시\n📌 할인 기간: 2/26(수) ~ 3/31(월)\n※ 자세한 내용은 포켓CU 이벤트 페이지를 참고해주세요!\n\n오직 CU에서 만나보세요💜\n도) 백종원스페셜한판 / 4,900원\n도) 백종원스페셜매콤불고기 / 4,900원\n주) 백종원스페셜우...","[백종원스페셜한판] 한식, 간편식, 백종원, 40% 할인, 집밥, 간편식\n[백종원스페셜매콤불고기] 매콤, 불고기, 백종원, 40% 할인, 집밥, 간편식","[도)백종원스